# Kinematics Calculator
This notebook is used for deriving and calculating the rig's kinematics

## Deriving the symbolic transformations Base -> End of Beam

In [12]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib
import sympy as sp
import plotly.io as pio

# Declaring the symbols
d1, theta_y, theta_p, a_b, h_b, camera_beam_offset, camera_height, camera_offset_from_beam_centre, camera_angle_deg = sp.symbols(
    r'd1 theta_y theta_p a_b h_b c_{boff} c_{h} c_{boc} \beta_{cam}',
    real=True
)

# Defining the DH parameters (with additional offsets on the links)
# Please note two more translation parameters are added to the DH table to account for the offsets in the links 
# The columns means as follows: [link number, link length, link twist, link offset (3 axes) joint angle]
DH = sp.Matrix([[1, 0, 0, 0, 0, d1, theta_y], \
            [2, 0, sp.pi/2, 0, 0, 0, theta_p], \
            [3, a_b, 0, 0, h_b, 0, 0], \
            [4, 0, 0, camera_beam_offset, camera_height, camera_offset_from_beam_centre, 0],
            [5, 0, sp.pi*camera_angle_deg/180, 0, 0, 0, sp.pi]])

DH


Matrix([
[1,   0,                  0,        0,     0,      d1, theta_y],
[2,   0,               pi/2,        0,     0,       0, theta_p],
[3, a_b,                  0,        0,   h_b,       0,       0],
[4,   0,                  0, c_{boff}, c_{h}, c_{boc},       0],
[5,   0, pi*\beta_{cam}/180,        0,     0,       0,      pi]])

In [13]:
# Calculating the individual transformation matrices using the DH parameters
N = len(DH[:, 0])

def create_transormation_string(superscript, subscript):
    return sp.symbols("{}^{" + str(superscript) + "}_" + str(subscript) + "T")


Full_forward_kinematics = sp.Matrix([[1, 0, 0, 0],\
                                  [0, 1, 0, 0],\
                                  [0, 0, 1, 0],
                                  [0, 0, 0, 1]])

Individual_transformation_matrices = []

for i in range(N):
    # Extracting parameters from the current link
    m = DH[i, 0]
    a_mm1 = DH[i, 1] #mm1 means m minus 1
    alpha_mm1 = DH[i, 2]
    tm = DH[i, 6]
    translate_i = DH[i, 3]
    translate_j = DH[i, 4]
    translate_k = DH[i, 5]



    print(f"=== LINK {m} ===")

    T_mm1_R = sp.Matrix([[1, 0, 0, 0], \
                         [0, sp.cos(alpha_mm1), -sp.sin(alpha_mm1), 0], \
                         [0, sp.sin(alpha_mm1), sp.cos(alpha_mm1), 0], \
                         [0, 0, 0, 1]])
    display(create_transormation_string(m-1, 'R'), T_mm1_R)

    T_R_P = sp.Matrix([[1, 0, 0, a_mm1], \
                       [0, 1, 0, 0], \
                       [0, 0, 1, 0], \
                       [0, 0, 0, 1]])
    display(create_transormation_string('R', 'P'), T_R_P)

    T_P_Q = sp.Matrix([[sp.cos(tm), -sp.sin(tm), 0, 0], \
                       [sp.sin(tm), sp.cos(tm), 0, 0], \
                       [0, 0, 1, 0], \
                       [0, 0, 0, 1]])   
    display(create_transormation_string('P', 'Q'), T_P_Q)

    T_Q_m = sp.Matrix([[1, 0, 0, translate_i], \
                       [0, 1, 0, translate_j], \
                       [0, 0, 1, translate_k], \
                       [0, 0, 0, 1]])
    display(create_transormation_string('Q', m), T_Q_m)

    # Calculating the transformation matrix from frame m-1 to frame m
    T_mm1_m = sp.simplify(T_mm1_R * T_R_P * T_P_Q * T_Q_m)
    Individual_transformation_matrices.append(T_mm1_m)

    Full_forward_kinematics = sp.simplify(Full_forward_kinematics * T_mm1_m)

    display(create_transormation_string(m-1, m), T_mm1_m)

print("=== Full Forward Kinematics ===")
display(sp.simplify(Full_forward_kinematics))

print("=== Individual Transformation Matrices ===")
display(Individual_transformation_matrices)



=== LINK 1 ===


{}^{0}_RT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

{}^{R}_PT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

...

Matrix([
[-1,                        0,                        0, 0],
[ 0, -cos(pi*\beta_{cam}/180), -sin(pi*\beta_{cam}/180), 0],
[ 0, -sin(pi*\beta_{cam}/180),  cos(pi*\beta_{cam}/180), 0],
[ 0,                        0,                        0, 1]])

=== Full Forward Kinematics ===


Matrix([
[-cos(theta_p)*cos(theta_y), sin(theta_p)*cos(theta_y)*cos(pi*\beta_{cam}/180) - sin(theta_y)*sin(pi*\beta_{cam}/180), sin(theta_p)*sin(pi*\beta_{cam}/180)*cos(theta_y) + sin(theta_y)*cos(pi*\beta_{cam}/180),  c_{boc}*sin(theta_y) + c_{boff}*cos(theta_p)*cos(theta_y) - c_{h}*sin(theta_p)*cos(theta_y) + (a_b*cos(theta_p) - h_b*sin(theta_p))*cos(theta_y)],
[-sin(theta_y)*cos(theta_p), sin(theta_p)*sin(theta_y)*cos(pi*\beta_{cam}/180) + sin(pi*\beta_{cam}/180)*cos(theta_y), sin(theta_p)*sin(theta_y)*sin(pi*\beta_{cam}/180) - cos(theta_y)*cos(pi*\beta_{cam}/180), -c_{boc}*cos(theta_y) + c_{boff}*sin(theta_y)*cos(theta_p) - c_{h}*sin(theta_p)*sin(theta_y) + (a_b*cos(theta_p) - h_b*sin(theta_p))*sin(theta_y)],
[             -sin(theta_p),                                                    -cos(theta_p)*cos(pi*\beta_{cam}/180),                                                    -sin(pi*\beta_{cam}/180)*cos(theta_p),                                                             a_b*sin(

=== Individual Transformation Matrices ===


[Matrix([
 [cos(theta_y), -sin(theta_y), 0,  0],
 [sin(theta_y),  cos(theta_y), 0,  0],
 [           0,             0, 1, d1],
 [           0,             0, 0,  1]]),
 Matrix([
 [cos(theta_p), -sin(theta_p),  0, 0],
 [           0,             0, -1, 0],
 [sin(theta_p),  cos(theta_p),  0, 0],
 [           0,             0,  0, 1]]),
 Matrix([
 [1, 0, 0, a_b],
 [0, 1, 0, h_b],
 [0, 0, 1,   0],
 [0, 0, 0,   1]]),
 Matrix([
 [1, 0, 0, c_{boff}],
 [0, 1, 0,    c_{h}],
 [0, 0, 1,  c_{boc}],
 [0, 0, 0,        1]]),
 Matrix([
 [-1,                        0,                        0, 0],
 [ 0, -cos(pi*\beta_{cam}/180), -sin(pi*\beta_{cam}/180), 0],
 [ 0, -sin(pi*\beta_{cam}/180),  cos(pi*\beta_{cam}/180), 0],
 [ 0,                        0,                        0, 1]])]

In [35]:
# Creating a function

parameters = (d1, theta_y, theta_p, a_b, h_b, camera_beam_offset, camera_height, camera_offset_from_beam_centre, camera_angle_deg)

_var_d1 = 0.353
_var_theta_y = 0.0
_var_theta_p = 0.0
_var_a_b = 0.83
_var_h_b = 0.0761
_var_camera_beam_offset = -0.043
_var_camera_height = 0.029
_var_camera_offset_from_beam_centre = 0.04
_var_camera_angle_deg = 40

_ffk = sp.lambdify(parameters, Full_forward_kinematics, modules='numpy')

_fT01 = sp.lambdify(parameters, Individual_transformation_matrices[0], modules='numpy')
_fT12 = sp.lambdify(parameters, Individual_transformation_matrices[1], modules='numpy')
_fT23 = sp.lambdify(parameters, Individual_transformation_matrices[2], modules='numpy')
_fT34 = sp.lambdify(parameters, Individual_transformation_matrices[3], modules='numpy')
_ftT45 = sp.lambdify(parameters, Individual_transformation_matrices[4], modules='numpy')

# Example calculation
T_cam = _ffk(
    _var_d1,
    _var_theta_y,
    _var_theta_p,
    _var_a_b,
    _var_h_b,
    _var_camera_beam_offset,
    _var_camera_height,
    _var_camera_offset_from_beam_centre,
    _var_camera_angle_deg
)

print("=== Full Forward Kinematics ===")
print(T_cam)

# Checking
print("=== Checking the individual transformation matrices ===")

_fkin_args = (
    _var_d1,
    _var_theta_y,
    _var_theta_p,
    _var_a_b,
    _var_h_b,
    _var_camera_beam_offset,
    _var_camera_height,
    _var_camera_offset_from_beam_centre,
    _var_camera_angle_deg
)

T01 = _fT01(*_fkin_args)
T12 = _fT12(*_fkin_args)
T23 = _fT23(*_fkin_args)
T34 = _fT34(*_fkin_args)
T45 = _ftT45(*_fkin_args)

#Matrix multiplication
T_cam_check = T01 @ T12 @ T23 @ T34 @ T45

print(T_cam_check)
matrices_correct = False

if np.allclose(T_cam, T_cam_check):
    print("The individual transformation matrices are correct.")
    matrices_correct = True
else:
    raise ValueError("The individual transformation matrices are not correct. Please check the calculations.")


if matrices_correct:
    Base_coords = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

    Frame_coords = [Base_coords]

    # Calculating the coordinates of each frame
    F1_coords = T01 @ Base_coords
    F2_coords = T01 @ T12 @ Base_coords
    F3_coords = T01 @ T12 @ T23 @ Base_coords
    F4_coords = T01 @ T12 @ T23 @ T34 @ Base_coords
    Fcam_coords = T01 @ T12 @ T23 @ T34 @ T45 @ Base_coords

    Frame_coords.append(F1_coords)
    Frame_coords.append(F2_coords)
    Frame_coords.append(F3_coords)
    Frame_coords.append(F4_coords)
    Frame_coords.append(Fcam_coords)

# Plot frame origins and coordinate systems in 3D using Plotly
origins = np.array([frame[:3, 3] for frame in Frame_coords], dtype=float)
rotations = np.array([frame[:3, :3] for frame in Frame_coords], dtype=float)

fig = go.Figure()

# Plot frame origins
fig.add_trace(go.Scatter3d(
    x=origins[:, 0],
    y=origins[:, 1],
    z=origins[:, 2],
    mode='markers+lines',
    marker=dict(size=5),
    line=dict(color='black', width=2),
    name='Frame origins'
))

# Plot coordinate axes for each frame
axis_length = 0.1
axis_colors = ['red', 'green', 'blue']
axis_names = ['X axis', 'Y axis', 'Z axis']

for axis in range(3):
    x_values, y_values, z_values = [], [], []

    for origin, rotation in zip(origins, rotations):
        endpoint = origin + axis_length * rotation[:, axis]

        x_values.extend([origin[0], endpoint[0], None])
        y_values.extend([origin[1], endpoint[1], None])
        z_values.extend([origin[2], endpoint[2], None])

    fig.add_trace(go.Scatter3d(
        x=x_values,
        y=y_values,
        z=z_values,
        mode='lines',
        line=dict(color=axis_colors[axis], width=4),
        name=axis_names[axis]
    ))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='3D Plot of Forward Kinematics'
)

fig.show()

=== Full Forward Kinematics ===
[[-1.          0.          0.          0.787     ]
 [-0.          0.64278761 -0.76604444 -0.04      ]
 [-0.         -0.76604444 -0.64278761  0.4581    ]
 [ 0.          0.          0.          1.        ]]
=== Checking the individual transformation matrices ===
[[-1.          0.          0.          0.787     ]
 [ 0.          0.64278761 -0.76604444 -0.04      ]
 [ 0.         -0.76604444 -0.64278761  0.4581    ]
 [ 0.          0.          0.          1.        ]]
The individual transformation matrices are correct.


In [38]:
# Learning how to project points onto the camera

# Generating point tracks
Beam_radius = 0.83
Wheel_witdh = 0.1
inner_track_radius = Beam_radius - Wheel_witdh / 2
outer_track_radius = Beam_radius + Wheel_witdh / 2

outer_track_points = []
inner_track_points = []

for angle in np.linspace(0, 2 * np.pi, 300):
    inner_x = inner_track_radius * np.cos(angle)
    inner_y = inner_track_radius * np.sin(angle)
    outer_x = outer_track_radius * np.cos(angle)
    outer_y = outer_track_radius * np.sin(angle)

    inner_track_points.append([inner_x, inner_y, 0])
    outer_track_points.append([outer_x, outer_y, 0])

# Inverting the camera's transformation matrix to get the transformation from the camera to the base frame
T_cam_to_base = np.linalg.inv(T_cam)
print("=== Transformation from camera to base frame ===")
print(T_cam_to_base)

#TODO Have these import in code, these have been now looked up from the camera calibration steam
# Image resolution
resolution = np.array([640, 480])  # [width, height]

# Camera intrinsic matrix
K = np.array([
    [461.8016663,   0.0,         324.1998901, 0.0],
    [0.0,           461.7047424, 242.8716125, 0.0],
    [0.0,           0.0,           1.0, 0.0]
])

# Distortion coefficients: [k1, k2, t1, t2, k3]
dist_coeffs = np.array([
     0.0056061,
    -0.0481728,
    -0.00040035,
     0.00041010,
     0.0330434
])

# For this single point
track_points_in_camera_frame = []

for track_point in inner_track_points:
    track_point_in_base_frame = np.array([track_point[0], track_point[1], track_point[2], 1])
    track_point_in_camera_frame = T_cam_to_base @ track_point_in_base_frame
    track_points_in_camera_frame.append(track_point_in_camera_frame)

for track_point in outer_track_points:
    track_point_in_base_frame = np.array([track_point[0], track_point[1], track_point[2], 1])
    track_point_in_camera_frame = T_cam_to_base @ track_point_in_base_frame
    track_points_in_camera_frame.append(track_point_in_camera_frame)

track_points_in_camera_frame = np.array(track_points_in_camera_frame)
track_points_pixel_coords = []

for track_point_in_camera_frame in track_points_in_camera_frame:
    pixel_coords_homogeneous = K @ track_point_in_camera_frame
    if pixel_coords_homogeneous[2] <= 0:
        # Point is behind the camera, skip it
        continue
    pixel_coords = np.rint(pixel_coords_homogeneous[:2] / pixel_coords_homogeneous[2]).astype(int)
    # Check if the pixel coordinates are within the image resolution
    if 0 <= pixel_coords[0] < resolution[0] and 0 <= pixel_coords[1] < resolution[1]: 
        track_points_pixel_coords.append(pixel_coords)




=== Transformation from camera to base frame ===
[[-1.         -0.         -0.          0.787     ]
 [-0.          0.64278761 -0.76604444  0.37663646]
 [-0.         -0.76604444 -0.64278761  0.26381923]
 [ 0.          0.          0.          1.        ]]


In [39]:
# ============================================================
# VISUALISATION
# Uses the variables calculated above.
# Does NOT modify the projection / kinematics calculations.
# ============================================================

import numpy as np
import plotly.graph_objects as go


# ============================================================
# PREPARE DATA
# ============================================================

frame_names = ["Base", "F1", "F2", "F3", "F4", "Camera"]

# Homogeneous transforms of all robot frames
frame_mats = [
    np.asarray(T, dtype=float)
    for T in Frame_coords
]

# Frame origins in base coordinates
origins_vis = np.vstack([
    T[:3, 3]
    for T in frame_mats
])

# Track points in base coordinates
inner_track_3d = np.asarray(
    inner_track_points,
    dtype=float
)

outer_track_3d = np.asarray(
    outer_track_points,
    dtype=float
)

# Valid projected pixels calculated in your previous cell
track_pixels_vis = np.asarray(
    track_points_pixel_coords,
    dtype=float
)

width = int(resolution[0])
height = int(resolution[1])


# ============================================================
# 1. 3D VIEW
# ROBOT FRAMES + INNER AND OUTER WHEEL TRACKS
# ============================================================

fig_3d = go.Figure()


# ------------------------------------------------------------
# Kinematic chain
# ------------------------------------------------------------

fig_3d.add_trace(go.Scatter3d(
    x=origins_vis[:, 0],
    y=origins_vis[:, 1],
    z=origins_vis[:, 2],

    mode="lines+markers+text",

    text=frame_names,
    textposition="top center",

    marker=dict(size=5),
    line=dict(width=4),

    name="Kinematic chain"
))


# ------------------------------------------------------------
# Coordinate axes of each frame
# ------------------------------------------------------------

axis_length = 0.10

axis_specs = [
    (0, "red",   "X axes"),
    (1, "green", "Y axes"),
    (2, "blue",  "Z axes")
]

for axis_idx, color, label in axis_specs:

    xs = []
    ys = []
    zs = []

    for T in frame_mats:

        origin = T[:3, 3]

        # Columns of the rotation matrix are the
        # X, Y and Z axes of that frame
        direction = T[:3, axis_idx]

        endpoint = origin + axis_length * direction

        xs.extend([
            origin[0],
            endpoint[0],
            None
        ])

        ys.extend([
            origin[1],
            endpoint[1],
            None
        ])

        zs.extend([
            origin[2],
            endpoint[2],
            None
        ])

    fig_3d.add_trace(go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,

        mode="lines",

        line=dict(
            color=color,
            width=5
        ),

        name=label
    ))


# ------------------------------------------------------------
# Inner wheel track
# ------------------------------------------------------------

fig_3d.add_trace(go.Scatter3d(
    x=inner_track_3d[:, 0],
    y=inner_track_3d[:, 1],
    z=inner_track_3d[:, 2],

    mode="lines",

    line=dict(width=4),

    name="Inner wheel track"
))


# ------------------------------------------------------------
# Outer wheel track
# ------------------------------------------------------------

fig_3d.add_trace(go.Scatter3d(
    x=outer_track_3d[:, 0],
    y=outer_track_3d[:, 1],
    z=outer_track_3d[:, 2],

    mode="lines",

    line=dict(width=4),

    name="Outer wheel track"
))


# ------------------------------------------------------------
# Highlight camera origin
# ------------------------------------------------------------

camera_origin = frame_mats[-1][:3, 3]

fig_3d.add_trace(go.Scatter3d(
    x=[camera_origin[0]],
    y=[camera_origin[1]],
    z=[camera_origin[2]],

    mode="markers+text",

    text=["Camera"],
    textposition="bottom center",

    marker=dict(size=8),

    name="Camera"
))


# ------------------------------------------------------------
# 3D plot formatting
# ------------------------------------------------------------

fig_3d.update_layout(

    title="Robot kinematic frames and wheel track",

    scene=dict(
        xaxis_title="X [m]",
        yaxis_title="Y [m]",
        zaxis_title="Z [m]",

        # Keep real-world proportions
        aspectmode="data"
    )
)

fig_3d.show()


# ============================================================
# 2. RGB CAMERA FRAME + VALID PROJECTED TRACK POINTS
# ============================================================

fig_rgb = go.Figure()


# ------------------------------------------------------------
# RGB image boundary
# ------------------------------------------------------------

fig_rgb.add_trace(go.Scatter(

    x=[
        0,
        width,
        width,
        0,
        0
    ],

    y=[
        0,
        0,
        height,
        height,
        0
    ],

    mode="lines",

    line=dict(width=3),

    name=f"RGB frame ({width} x {height})"
))


# ------------------------------------------------------------
# Principal point
# ------------------------------------------------------------

cx_vis = float(K[0, 2])
cy_vis = float(K[1, 2])

fig_rgb.add_trace(go.Scatter(

    x=[cx_vis],
    y=[cy_vis],

    mode="markers+text",

    text=["Principal point"],
    textposition="top center",

    marker=dict(size=8),

    name="Principal point"
))


# ------------------------------------------------------------
# Valid projected track points
# ------------------------------------------------------------

if len(track_pixels_vis) > 0:

    fig_rgb.add_trace(go.Scatter(

        x=track_pixels_vis[:, 0],
        y=track_pixels_vis[:, 1],

        # Use markers rather than lines because your existing
        # calculation removes points that are outside the image.
        mode="markers",

        marker=dict(size=4),

        name="Projected wheel track"
    ))


# ------------------------------------------------------------
# Image-coordinate formatting
#
# Camera image convention:
#
# (0,0) --------> u
#   |
#   |
#   v
#
# Therefore y must increase downwards.
# ------------------------------------------------------------

fig_rgb.update_layout(

    title="Wheel track projected onto RGB camera frame",

    xaxis=dict(
        title="u [pixels]",
        range=[0, width],

        scaleanchor="y",
        scaleratio=1
    ),

    yaxis=dict(
        title="v [pixels]",
        range=[height, 0]
    )
)

fig_rgb.show()


# ============================================================
# 3. NUMERICAL SUMMARY
# ============================================================

print("=== Projection summary ===")

print(
    f"Inner track points generated: "
    f"{len(inner_track_points)}"
)

print(
    f"Outer track points generated: "
    f"{len(outer_track_points)}"
)

print(
    f"Total 3D track points: "
    f"{len(inner_track_points) + len(outer_track_points)}"
)

print(
    f"Points transformed to camera frame: "
    f"{len(track_points_in_camera_frame)}"
)

print(
    f"Valid points projected inside RGB image: "
    f"{len(track_points_pixel_coords)}"
)

print(
    f"RGB resolution: "
    f"{width} x {height}"
)

=== Projection summary ===
Inner track points generated: 300
Outer track points generated: 300
Total 3D track points: 600
Points transformed to camera frame: 600
Valid points projected inside RGB image: 101
RGB resolution: 640 x 480
